In [6]:
"""
Greedy algorithms typically have this top-down design:
make a choise and then solve a subproblem, rather than
the bottom-up technique of solving subproblems before making
a choice.

stores the indexes of the
start time of candidate ≥ finish time of last selected

activity_selector example:
s = [0, 1, 3, 0, 5, 3, 5, 6, 8, 8, 2, 12]
f = [0, 4, 5, 6, 7, 9, 9, 10, 11, 12, 14, 16]
[1] + [4] + [8] + [11] + []
1 > 0, 5 > 4, 8 > 7, 12 > 11

simply picks times that don't overlap
"""
def recursive_activity_selector(s, f, k):
    m = k + 1
    
    while m < len(s) and s[m] < f[k]:
        m += 1
    
    if m < len(s):
        return [m] + recursive_activity_selector(s, f, m)
    else:
        return []

In [4]:
def greedy_activity_selector(s, f):
    selected = [0]
    k = 0
    
    for m in range(1, len(s)):
        if s[m] >= f[k]:
            selected.append(m)
            k = m
            
    return selected

In [5]:
def greedy_activity_selector_clean(activities):
    activities = sorted(activities, key=lambda x: x[1])
    
    selected = [activities[0]]
    last_finish = activities[0][1]
    
    for start, finish in activities[1:]:
        if start >= last_finish:
            selected.append((start, finish))
            last_finish = finish
            
    return selected

In [ ]:
"""
The DP solution proves correctness structure:
Optimal solution = optimal left + optimal right + chosen activity

The greedy solution exploits a structural property:
Choosing the activity that finishes earliest always leaves maximum room for the rest.

That property collapses O(n³) → O(n).
The maximum number of mutually compatible activities we can select
c[i][j] = maximum number of non-overlapping activities
          that fit strictly between activity i and activity j

Imagine this:
i -------------------- j

We’re trying to pack activities between those boundaries.
For each possible choice k:
i ---- k ---- j

We split into two independent subproblems:
Left side: (i, k)
Right side: (k, j)
Then combine:
best_left + best_right + 1

So the DP table stores:
The size of the best schedule in every possible sub-interval.
"""
def activity_selector_dp(s, f):
    n = len(s)
    
    c = [[0] * n for _ in range(n)]
    solution = [[None] * n for _ in range(n)]
    
    # gap = distance between i and j
    for gap in range(2, n):
        for i in range(n - gap):
            j = i + gap
            
            for k in range(i + 1, j):
                if f[i] <= s[k] and f[k] <= s[j]:
                    value = c[i][k] + c[k][j] + 1
                    
                    if value > c[i][j]:
                        c[i][j] = value
                        solution[i][j] = k
    
    return c, solution